### Imports

In [1]:
from fundus_dataset import AugmentPair, FundusVesselDataset
from torch.utils.data import DataLoader
import torch.nn as nn
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Subset
from monai.networks.nets import UNet
from monai.losses import DiceLoss
from safetensors.torch import save_file
from pathlib import Path
from common_utils import get_datasets, get_dataloaders, train_model, make_unet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


### Testing the Dataset

In [2]:
img_dir = "fundus/train/Original/"
mask_dir = "fundus/train/Ground truth"

train_full, val_full = get_datasets(img_dir, mask_dir, transform=AugmentPair(crop_size=(512, 512)))
train_loader, val_loader = get_dataloaders(train_full, val_full, batch_size=8, num_workers=8)


## Sanity checking
images, masks = next(iter(train_loader))
val_images, val_masks = next(iter(val_loader))

print("Train images:", images.shape, images.dtype, images.min().item(), images.max().item())
print("Train masks: ", masks.shape, masks.dtype, torch.unique(masks))
print("Val images:", val_images.shape, val_images.dtype, val_images.min().item(), val_images.max().item())
print("Val masks: ", val_masks.shape, val_masks.dtype, torch.unique(val_masks))

Number of images: 600
Number of masks: 600
First 5 image files:  ['100_A.png', '101_A.png', '102_A.png', '103_A.png', '104_A.png']
First 5 mask files:  ['100_A.png', '101_A.png', '102_A.png', '103_A.png', '104_A.png']
Number of images: 600
Number of masks: 600
First 5 image files:  ['100_A.png', '101_A.png', '102_A.png', '103_A.png', '104_A.png']
First 5 mask files:  ['100_A.png', '101_A.png', '102_A.png', '103_A.png', '104_A.png']
Train size: 480
Val size: 120


Train images: torch.Size([8, 3, 512, 512]) torch.float32 0.0 0.8901960849761963
Train masks:  torch.Size([8, 1, 512, 512]) torch.float32 tensor([0., 1.])
Val images: torch.Size([1, 3, 2048, 2048]) torch.float32 0.0 1.0
Val masks:  torch.Size([1, 1, 2048, 2048]) torch.float32 tensor([0., 1.])


### BCEDice Loss Functions

In [3]:
class BCEDiceLoss(nn.Module):
    def __init__(self, w_bce=0.5, w_dice=0.5):
        super().__init__()
        self.w_bce = w_bce
        self.w_dice = w_dice

        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss(sigmoid=True)

    def forward(self, logits, masks):
        bce = self.bce(logits, masks)
        dice = self.dice(logits, masks)

        return self.w_bce * bce + self.w_dice * dice

### DANGEROUS (RESET EXPERIMENT)

In [4]:
model = make_unet(channels=(32, 64, 128, 256, 512))
loss_fn = BCEDiceLoss(w_bce=0.5, w_dice=0.5).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 100
SAVE_PATH = "/workspace/models_baseline/"
SAVE_NAME = "baseline.safetensors"

### START OR CONTINUE EXPERIMENT

In [5]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    save_path=SAVE_PATH,
    save_name=SAVE_NAME,
)

Epoch 001/100 | Train loss: 0.7735 | Val loss: 0.7662 | Val Dice: 0.1649 | Best: 0.1649 @ 1
Epoch 002/100 | Train loss: 0.7224 | Val loss: 0.7279 | Val Dice: 0.4957 | Best: 0.4957 @ 2
Epoch 003/100 | Train loss: 0.6934 | Val loss: 0.7001 | Val Dice: 0.6529 | Best: 0.6529 @ 3
Epoch 004/100 | Train loss: 0.6566 | Val loss: 0.6599 | Val Dice: 0.6498 | Best: 0.6529 @ 3
Epoch 005/100 | Train loss: 0.6202 | Val loss: 0.6440 | Val Dice: 0.5984 | Best: 0.6529 @ 3
Epoch 006/100 | Train loss: 0.5796 | Val loss: 0.5770 | Val Dice: 0.7030 | Best: 0.7030 @ 6
Epoch 007/100 | Train loss: 0.5304 | Val loss: 0.5291 | Val Dice: 0.7841 | Best: 0.7841 @ 7
Epoch 008/100 | Train loss: 0.4837 | Val loss: 0.5048 | Val Dice: 0.7815 | Best: 0.7841 @ 7
Epoch 009/100 | Train loss: 0.4515 | Val loss: 0.4399 | Val Dice: 0.8022 | Best: 0.8022 @ 9
Epoch 010/100 | Train loss: 0.3995 | Val loss: 0.4042 | Val Dice: 0.8133 | Best: 0.8133 @ 10
Epoch 011/100 | Train loss: 0.3661 | Val loss: 0.3783 | Val Dice: 0.8152 | Best

### Save your Settings

In [6]:
import pandas as pd

history_df = pd.DataFrame(history)
history_df.to_csv("/workspace/models_baseline/baseline_history.csv", index=False)